## OpenAI API Version 2 and later

### Installing Dependencies

You can install all required packages at once using the `requirements.txt` file (recommended), or install them individually.

In [11]:
pip install -r ./requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [9]:
# Alternative: install packages individually (useful for quick experimentation).
# python-dotenv is used to load the API key safely from a .env file.
%pip install openai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Verify the installed version of the openai package.
%pip show openai

Name: openai
Version: 2.26.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /home/dda/projects/python/Claude-Code-Projects/.venv/lib/python3.13/site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: langchain-openai, litellm, semantic-router
Note: you may need to restart the kernel to use updated packages.


#### Chat Completions API With OpenAI Version 2 or later

In [18]:
import openai
import os
from dotenv import load_dotenv

# Load environment variables from a .env file located in the current directory.
# The .env file should contain a line like: OPENAI_API_KEY=your-key-here
# Never hardcode API keys in source code — they can be accidentally shared or committed.
load_dotenv()

from openai import OpenAI

# Instantiate the OpenAI client.
# It automatically reads OPENAI_API_KEY from the environment (set by load_dotenv above).
client = OpenAI()

# Send a chat completion request to the API.
# 'messages' is a list of conversation turns, each with a 'role' and 'content'.
#   - 'system': sets the model's persona or behavior instructions
#   - 'user': represents what the human says
#   - 'assistant': represents a previous model reply (used for multi-turn conversations)
response = client.chat.completions.create(
    model="gpt-5.2",               # The model to use for generation
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Make a list with the largest 3 cities in Europe."}
    ],
    temperature=1,                  # Controls randomness: 0 = deterministic, 2 = very creative
    max_completion_tokens=1024      # Maximum number of tokens the model can generate in the reply
)

In [19]:
# Access the generated text from the response.
# response.choices is a list of outputs (usually just one unless n > 1 was set).
# .message.content holds the plain text of the assistant's reply.
print(response.choices[0].message.content)

- **Istanbul (Turkey)** – ~15+ million (largest city in Europe by total population; transcontinental)  
- **Moscow (Russia)** – ~12+ million (largest city entirely in Europe)  
- **London (United Kingdom)** – ~9 million


In [15]:
# Inspect the full response object returned by the API.
# Key fields to know:
#   - choices: list of generated outputs
#   - usage: token counts for prompt + completion (important for cost tracking)
#   - model: the exact model version that was used
#   - id: unique identifier for this request (useful for debugging or logging)
response

ChatCompletion(id='chatcmpl-DL7CGqG4NdNgS2lFrVuFZw83CCAt8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='- **Istanbul (Turkey)** — ~15+ million (city), ~16+ million (metro)  \n- **Moscow (Russia)** — ~12–13 million (city), ~17+ million (metro)  \n- **London (United Kingdom)** — ~9 million (city), ~14–15 million (metro)  \n\n*Note: Rankings can vary depending on whether you count city proper, urban area, or metro area, and whether Istanbul’s transcontinental location is included in “Europe.”*', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1773924924, model='gpt-5.2-2025-12-11', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=108, prompt_tokens=28, total_tokens=136, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_token

In [16]:
# GPT-5 family models can return rich text formatted as Markdown.
# We explicitly ask the model to use Markdown so we can render it nicely below.
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a talented tutor."},
        {"role": "user", "content": "Explain RAG. Use Markdown with headings, lists, and code blocks."}
    ]
)

In [17]:
# IPython's Markdown class renders the raw Markdown string as formatted HTML in the notebook.
# This is useful when the model's output is intended to be read as structured content
# with headings, bullet points, and code blocks.
from IPython.display import Markdown
Markdown(response.choices[0].message.content)

# Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system (which finds relevant text from a large knowledge base) with a generative model (which composes answers). The goal is to ground the generator in evidence, improving factuality, coverage of long-tail knowledge, and up-to-dateness.

## High-level workflow

1. User query
2. Retriever: find top-k relevant documents/passages from a datastore
3. Reader/Generator: condition a generative model on the retrieved passages + query to produce a final answer
4. (Optional) Reranker or verifier: reorder or filter candidate passages/answers; optionally check factual consistency

## Why use RAG?

- Improves factual accuracy by grounding answers in source documents
- Allows LLMs to use large external corpora without storing everything in model weights
- Makes outputs more interpretable (can return citations or snippets)
- Enables up-to-date knowledge without retraining the model

## Core components

- Document store: large collection of documents, split into passages/chunks
- Index: structure for fast lookup (dense vector index like FAISS/Annoy/HNSW, or sparse inverted index like BM25)
- Embeddings model: maps queries and documents into a vector space (dense retrieval)
- Retriever: uses embeddings or sparse methods to find top-k items
- Generator/Reader: LLM (e.g., GPT-family, Llama, or a fine-tuned seq2seq) that conditions on retrieved context
- Reranker/verifier (optional): further refines results or checks factuality

## Retrieval strategies

- Sparse retrieval (BM25, Lucene): fast, interpretable, no embeddings needed
- Dense retrieval (embedding + ANN): better semantic matching for paraphrases
- Hybrid: combine sparse + dense scores
- Two-stage retrieval: cheap retriever for recall, expensive reranker for precision

## Typical design choices

- Chunk size: 200–1,000 tokens commonly; overlapping windows help recall
- k (number of passages): 3–50 depending on LLM context window and noise tolerance
- Concatenation vs. multi-hop: include retrieved passages directly or use multiple generation steps
- Citation format: include snippet IDs or URLs in generation prompt to encourage grounded answers

## Example: simple Python-like RAG pipeline

This example uses conceptual code to show the flow (embeddings + FAISS + LLM). Replace with real libraries (sentence-transformers, faiss, OpenAI, transformers, LangChain, Chroma, Pinecone, etc.).

```python
# Pseudocode for a minimal RAG pipeline

# 1. Build document embeddings and index (done offline)
documents = ["doc text 1", "doc text 2", ...]
doc_ids = ["id1", "id2", ...]
embeddings = embed_model.embed(documents)            # e.g., sentence-transformers
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)                                # add vectors in same order as doc_ids

# 2. Query-time retrieval
query = "How does photosynthesis work?"
q_vec = embed_model.embed([query])[0]
D, I = index.search(np.array([q_vec]), k=5)         # top-k doc indices
retrieved_docs = [documents[i] for i in I[0]]
retrieved_ids = [doc_ids[i] for i in I[0]]

# 3. Build prompt for LLM
context = "\n\n--\n\n".join(f"[{id}] {text}" for id, text in zip(retrieved_ids, retrieved_docs))
prompt = f"""Use the documents below to answer the question. Cite doc ids in your answer when you use them.

Documents:
{context}

Question: {query}

Answer (include citations like [id1], keep answer concise):
"""

# 4. Generate
answer = llm.generate(prompt)  # call to GPT / Llama / etc.
print(answer)
```

## Example using LangChain (conceptual)

```python
from langchain.chains import RetrievalQA
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.llms import OpenAI

# Set up embedding model and vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(doc_texts, embeddings)

# Build a retrieval-augmented QA chain
llm = OpenAI(temperature=0)
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())

# Run
result = qa.run("What is transfer learning?")
print(result)
```

Notes:
- "stuff" concatenates all retrieved passages into the prompt. Other strategies like "map_reduce" or "refine" can be used for longer corpora or multi-step reasoning.
- Use temperature=0 for more deterministic answers when factual accuracy matters.

## Variants of RAG

- RAG-Token / RAG-Sequence (original paper): generator attends to retrieved documents at token generation time vs. using combined context
- Fusion-in-Decoder (FiD): encode each retrieved passage separately and fuse in the decoder for improved performance
- Multi-hop RAG: chain retrieval steps where the answer to one retrieval informs the next

## Evaluation metrics

- Retrieval: recall@k, precision@k, MRR
- End-to-end QA: exact match (EM), F1, ROUGE, BLEU (depending on task)
- Human evaluation: factuality, helpfulness, fluency, citation correctness

## Best practices

- Use overlapping chunks to increase recall across boundaries
- Tune k to balance context noise vs. missing evidence
- Use rerankers or cross-encoders for improved precision after coarse retrieval
- Prefer deterministic generation (low temperature) and explicit citation prompts for factual tasks
- Keep prompt length below model context limit; consider FiD-like architectures for many documents

## Risks and limitations

- Hallucination: generator can still invent facts not present in retrieved passages
- Garbage-in, garbage-out: poor retrieval yields poor grounding
- Latency/cost: retrieval, embedding, and generation add runtime and compute cost
- Privacy and data freshness: ensure the document store is maintained and access-controlled

## Practical tips for reducing hallucinations

- In the prompt, instruct the model to only answer when evidence exists and otherwise say "I don't know"
- Provide retrieved snippets and ask the model to cite passages
- Use a verifier: check claims against the retrieved documents with a separate classification model
- Use conservative generation parameters (temperature=0, beam search / deterministic decoding)

## Quick checklist to implement RAG

- [ ] Prepare and clean a document corpus
- [ ] Split into passages and choose chunk size + overlap
- [ ] Choose embedding model and build index (FAISS, HNSW, Pinecone, Chroma, etc.)
- [ ] Implement retriever with top-k retrieval
- [ ] Decide generation strategy: prompt concatenation or FiD-style encoder-decoder
- [ ] Add reranker/verifier if needed
- [ ] Add tooling for citations and provenance
- [ ] Monitor metrics and human-review outputs

If you want, I can:
- Show a runnable example with a specific library (FAISS + sentence-transformers + Hugging Face Transformers)
- Draft prompt templates to reduce hallucination and improve citation behavior
- Suggest index and chunking settings tailored to your corpus size and latency constraints. Which would you prefer?